In [ ]:
# 
# import dias.rewriter
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import os
# STEFANOS: Conditionally import Modin Pandas
if "IREWR_WITH_MODIN" in os.environ and os.environ["IREWR_WITH_MODIN"] == "True":
    # STEFANOS: Import Modin Pandas
    import os
    os.environ["MODIN_ENGINE"] = "ray"
    import ray
    ray.init(num_cpus=int(os.environ['MODIN_CPUS']), runtime_env={'env_vars': {'__MODIN_AUTOIMPORT_PANDAS__': '1'}})
    import modin.pandas as pd
else:
    # STEFANOS: Import regular Pandas
    import pandas as pd
import seaborn as sns  # data visualization
import matplotlib.pyplot as plt


# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/home/dias-benchmarks/notebooks/ampiiere/animal-crossing-villager-popularity-analysis/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
### cell 0 ###

vlgr_df = pd.read_csv("/home/dias-benchmarks/notebooks/ampiiere/animal-crossing-villager-popularity-analysis/input/animal-crossing-new-horizons-nookplaza-dataset/villagers.csv")
popul_df = pd.read_csv("/home/dias-benchmarks/notebooks/ampiiere/animal-crossing-villager-popularity-analysis/input/acnh-villager-popularity/acnh_villager_data.csv")
factor = 400
popul_df = pd.concat([popul_df]*factor, ignore_index=True)
vlgr_df = pd.concat([vlgr_df]*factor, ignore_index=True)
print(popul_df.info())
vlgr_df.info()

In [ ]:
### cell 1 ###

vlgr_df.head()

In [ ]:
### cell 2 ###

popul_df.head()

In [ ]:
### cell 3 ###

vlgr_df.info()

In [ ]:
### cell 4 ###

popul_df.info()

In [ ]:
### cell 5 ###

# There are some missing/non-matching names 
vlgr_df["Name"].isin(popul_df['name']).sum()

In [ ]:
### cell 6 ###

# Optimized: use boolean inversion and single column selection
mismatch_names = popul_df.loc[~popul_df["name"].isin(vlgr_df["Name"]), "name"]
mismatch_names

In [ ]:
### cell 7 ###

popul_df['name'].replace({
    'OHare': "O'Hare",
    'Buck(Brows)': 'Buck',
    'Renee': 'Renée',
    'WartJr': 'Wart Jr.',
    'Crackle(Spork)': 'Spork'
}, inplace=True)

In [ ]:
### cell 8 ###

# Checking if All names match
vlgr_df["Name"].isin(popul_df['name']).sum()

In [ ]:
### cell 9 ###

popul_df = popul_df[popul_df['name'].isin(vlgr_df['Name'])]

In [ ]:
### cell 10 ###

# Now that both df have same length, we can set index as names and combine the 2 dfs
popul_df.set_index('name', drop=True, inplace=True)
vlgr_df.set_index('Name', drop=True, inplace=True)

In [ ]:
### cell 11 ###

combined_df = popul_df.merge(vlgr_df, left_index=True, right_index=True)

In [ ]:
### cell 12 ###

# drop irrelevent columns
combined_df.drop(columns=['Furniture List', 'Filename', 'Unique Entry ID', "Wallpaper", "Flooring", "Birthday", "Favorite Song"], inplace=True)

In [ ]:
### cell 13 ###

# Optimized: directly insert the new ranking column at position 2
df = combined_df  # alias for brevity

df.sort_values(['tier', 'rank'], inplace=True)
df.insert(2, 'overall_ranking', np.arange(1, df.shape[0] + 1))

In [ ]:
### cell 14 ###

overall_mean = combined_df.overall_ranking.mean()
print(f'The overall_mean is {overall_mean}, this would serve as a baseline for to compare against popularity performance of our features.')

In [ ]:
### cell 15 ###

combined_df.columns

In [ ]:
### cell 16 ###

combined_df['Gender'].value_counts()

In [ ]:
### cell 17 ###

combined_df.groupby('tier').Gender.value_counts()

In [ ]:
### cell 18 ###

pd.pivot_table(combined_df, index = 'tier', values = 'Catchphrase', columns="Gender", aggfunc='count')

In [ ]:
### cell 19 ###

species_ranking = combined_df.groupby('Species', as_index=False)['overall_ranking'].mean().sort_values('overall_ranking')

In [ ]:
### cell 20 ###

combined_df.Personality.value_counts()

In [ ]:
### cell 21 ###

personality_ranking = (
    combined_df
        .groupby('Personality', sort=False)['overall_ranking']
        .mean()
        .sort_values()
        .reset_index()
)

In [ ]:
### cell 22 ###

combined_df.groupby(['tier','Personality'])['Catchphrase'].count().unstack('Personality')

In [ ]:
### cell 23 ###

# Only select the columns we need, use as_index=False to avoid extra reset_index, and chain operations
style_ranking1 = (
    combined_df[['Style 1', 'overall_ranking']]
    .groupby('Style 1', as_index=False)['overall_ranking']
    .mean()
    .sort_values('overall_ranking')
)
style_ranking2 = (
    combined_df[['Style 2', 'overall_ranking']]
    .groupby('Style 2', as_index=False)['overall_ranking']
    .mean()
    .sort_values('overall_ranking')
)

In [ ]:
### cell 24 ###

# Compute the average using NumPy arrays to minimize pandas-overhead
avg = (style_ranking1['overall_ranking'].values + style_ranking2['overall_ranking'].values) * 0.5

# Make a single copy of the frame and assign in place
style_ranking = style_ranking1.copy()
style_ranking['overall_ranking'] = avg

In [ ]:
### cell 25 ###

style_ranking

In [ ]:
### cell 26 ###

combined_df.groupby(['tier', 'Style 1'], sort=True)['Catchphrase'].count().unstack('Style 1')

In [ ]:
### cell 27 ###

# Vectorized equivalent of pd.pivot_table(..., aggfunc='count')
combined_df.groupby(['tier', 'Style 2'])['Catchphrase'].count().unstack('Style 2')